In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M15.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7719803978469849, 'n_it': 0.3935495060487712}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 700

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[13.708146770197766, 13.608391012881276, 14.516994823863353, 14.641527068237759, 13.305927994505183, 15.993325771187374, 16.1700976552358, 17.94682694988638, 17.21228271956889, 15.75398878637472, 14.469569103454976, 14.065350729867081, 14.84370019714221, 14.43834206075853, 16.878641447845343, 16.063631907065922, 13.742407674482921, 16.661882871709686, 13.786783907193046, 13.772060536576928, 13.810093713272819, 13.757388961232984, 15.133566896978637, 15.274418675773362, 13.682524221258774, 14.682203841211196, 14.01785278733381, 15.116795944824892, 17.50883247106341, 13.80940821377668, 13.861960224196329, 13.981074681781696, 13.459358610547667, 15.131580100175572, 13.641063412064439, 13.746347319934952, 13.644934327956317, 13.456017278313546, 14.238051492705058, 13.38904775686515, 13.655765110571386, 13.906600523053644, 14.652436294900122, 16.62908488366938, 15.133932553771084, 13.382922897508225, 13.789847374240448, 16.597043551580697, 17.37808326708438, 15.312846745837557, 15.843788920

In [5]:
np.average(y_max_arr)

np.float64(14.89877722035077)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M15/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)